# Notebook 07 — Cascade Prediction Model

**Goal:** Given a delay event at station X (time, severity, network position),
predict whether the delay will cascade to downstream stations.

**Prediction target (binary):**
- Label = 1: delay at station A + ≥2 Granger-downstream stations also delayed within 3 hours
- Label = 0: isolated delay, no cascade propagation detected

**Models:** Logistic Regression (baseline) → Random Forest → Gradient Boosting

**Primary metric:** AUC-ROC (model comparison), Recall (operational priority)

**Data:** 4 seasonal strategic parquets (2024) + processed outputs from notebooks 03–05

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import pickle
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

ROOT     = Path('..').resolve()
PROC_DIR = ROOT / 'data' / 'processed'
(ROOT / 'web' / 'assets' / 'charts').mkdir(parents=True, exist_ok=True)

print('Imports OK')

## 2. Build Delay Events Dataset

In [ ]:
# Load 4 seasonal strategic parquets
OTP_COLS = ['route_id', 'line', 'parent_station', 'stop_timestamp',
            'scheduled_arrival_time', 'service_date', 'headway_branch_seconds']

SEASONS = ['2024-01', '2024-04', '2024-07', '2024-10']
frames = []
for ym in SEASONS:
    tmp = pd.read_parquet(PROC_DIR / 'strategic' / f'{ym}.parquet', columns=OTP_COLS)
    tmp['ym'] = ym
    frames.append(tmp)
raw = pd.concat(frames, ignore_index=True)

# Compute delay_sec (same conversion as nb06)
raw = raw.dropna(subset=['stop_timestamp', 'scheduled_arrival_time']).copy()
raw['midnight_unix'] = pd.to_datetime(raw['service_date']).apply(
    lambda d: pd.Timestamp(d, tz='America/New_York').timestamp()
)
raw['scheduled_unix'] = raw['midnight_unix'] + raw['scheduled_arrival_time']
raw['delay_sec']      = raw['stop_timestamp'] - raw['scheduled_unix']
raw = raw[raw['delay_sec'].abs() <= 3600].copy()

# Time features
raw['hour']        = (raw['scheduled_arrival_time'] // 3600).astype(int).clip(0, 23)
raw['service_date'] = pd.to_datetime(raw['service_date'])
raw['day_of_week'] = raw['service_date'].dt.dayofweek
raw['is_weekend']  = (raw['day_of_week'] >= 5)

print(f'Raw stop events: {len(raw):,}')
print(f'Date range     : {raw["service_date"].min().date()} → {raw["service_date"].max().date()}')
print(f'Unique stations: {raw["parent_station"].nunique()}')

In [ ]:
# Aggregate to (service_date, parent_station, hour) level
station_hour = (
    raw.groupby(['service_date', 'parent_station', 'line', 'hour', 'day_of_week', 'is_weekend', 'ym'])
    .agg(
        mean_delay  = ('delay_sec', 'mean'),
        max_delay   = ('delay_sec', 'max'),
        n_trips     = ('delay_sec', 'count'),
    )
    .reset_index()
)

# Delay event: mean delay > 5 min
DELAY_THRESHOLD = 300
station_hour['is_delay_event'] = station_hour['mean_delay'] > DELAY_THRESHOLD

# Delay severity: 0=none, 1=mild(5-10min), 2=moderate(10-20min), 3=severe(>20min)
bins   = [-np.inf, 300, 600, 1200, np.inf]
labels = [0, 1, 2, 3]
station_hour['delay_severity'] = pd.cut(
    station_hour['mean_delay'], bins=bins, labels=labels
).astype(int)

total  = len(station_hour)
n_del  = station_hour['is_delay_event'].sum()
print(f'Station-hour records : {total:,}')
print(f'Delay events (>300s) : {n_del:,}  ({n_del/total*100:.1f}%)')

## 3. Label Cascade Events

**Definition:** A delay event at station A on date D hour H is labeled **cascade=1** if:
- ≥ 2 Granger-downstream stations of A also had delay events (mean_delay > 300s)
  within the 3-hour window [H, H+3] on the same day.

Downstream stations are taken from `cascade_graph.parquet` (Granger causality edges from nb05).

In [ ]:
# Load Granger edges
cascade_graph = pd.read_parquet(PROC_DIR / 'cascade_graph.parquet')

# Build downstream lookup: src → set of dst stations
downstream_dict = cascade_graph.groupby('src')['dst'].apply(set).to_dict()
print(f'Granger edges    : {len(cascade_graph):,}')
print(f'Source stations  : {len(downstream_dict)}')

# Build fast lookup: (service_date, parent_station) → set of delayed hours
delay_hours = (
    station_hour[station_hour['is_delay_event']]
    .groupby(['service_date', 'parent_station'])['hour']
    .apply(set)
    .to_dict()
)
print(f'(date, station) with delays: {len(delay_hours):,}')

# Label cascade events
CASCADE_WINDOW = 3   # hours after source delay
CASCADE_MIN_DST = 2  # minimum downstream stations affected

def is_cascade(row):
    src      = row['parent_station']
    date     = row['service_date']
    hour     = row['hour']
    dsts     = downstream_dict.get(src, set())
    if not dsts:
        return 0
    affected = 0
    window   = set(range(hour, min(hour + CASCADE_WINDOW + 1, 24)))
    for dst in dsts:
        dst_hours = delay_hours.get((date, dst), set())
        if dst_hours & window:   # any overlap
            affected += 1
    return int(affected >= CASCADE_MIN_DST)

# Only label actual delay events (non-delay events are cascade=0 by definition)
delay_events = station_hour[station_hour['is_delay_event']].copy()
print(f'\nLabeling {len(delay_events):,} delay events...')
delay_events['cascade'] = delay_events.apply(is_cascade, axis=1)

n_cas = delay_events['cascade'].sum()
print(f'Cascade events   : {n_cas:,}  ({n_cas/len(delay_events)*100:.1f}% of delay events)')

## 4. Feature Engineering

In [ ]:
# Load station-level static features
cascade_scores = pd.read_parquet(PROC_DIR / 'station_cascade_score.parquet',
                                  columns=['parent_station', 'betweenness',
                                           'bunching_rate', 'n_events', 'out_degree'])

# Rename to avoid confusion with dynamic features
cascade_scores = cascade_scores.rename(columns={
    'bunching_rate': 'hist_bunching_rate',
    'n_events'     : 'station_volume',
    'out_degree'   : 'granger_out_degree',
})

# Normalise station_volume
cascade_scores['station_volume_norm'] = (
    cascade_scores['station_volume'] / cascade_scores['station_volume'].max()
)

# Build feature matrix
feat = delay_events[[
    'service_date', 'parent_station', 'line', 'ym',
    'hour', 'day_of_week', 'is_weekend',
    'delay_severity', 'n_trips', 'cascade'
]].copy()

# Time features
feat['is_morning_peak'] = feat['hour'].between(7, 9).astype(int)
feat['is_evening_peak'] = feat['hour'].between(16, 19).astype(int)
feat['is_peak']         = (feat['is_morning_peak'] | feat['is_evening_peak']).astype(int)
feat['is_weekend']      = feat['is_weekend'].astype(int)

# Cyclic hour encoding
feat['hour_sin'] = np.sin(2 * np.pi * feat['hour'] / 24)
feat['hour_cos'] = np.cos(2 * np.pi * feat['hour'] / 24)

# Join static station features
feat = feat.merge(cascade_scores, on='parent_station', how='left')
feat['betweenness']         = feat['betweenness'].fillna(0)
feat['hist_bunching_rate']  = feat['hist_bunching_rate'].fillna(0)
feat['granger_out_degree']  = feat['granger_out_degree'].fillna(0)
feat['station_volume_norm'] = feat['station_volume_norm'].fillna(0)

FEATURE_COLS = [
    'hour_sin', 'hour_cos',          # time (cyclic)
    'is_peak', 'is_weekend',          # time (categorical)
    'delay_severity',                  # how bad the delay is
    'betweenness',                     # network centrality
    'hist_bunching_rate',              # station's bunching history
    'granger_out_degree',              # historical cascade footprint
    'station_volume_norm',             # traffic volume proxy
    'n_trips',                         # trips in this hour
]

print(f'Feature matrix: {feat.shape}')
print(f'Cascade rate  : {feat["cascade"].mean():.3f}')
print(f'\nFeature columns: {FEATURE_COLS}')
print(f'\nMissing values:\n{feat[FEATURE_COLS].isnull().sum()}')

## 5. Train / Test Split (Time-Based)

In [ ]:
# Time-based split: train on Jan+Apr, test on Jul+Oct
# NEVER random shuffle time-series data
TRAIN_YM = ['2024-01', '2024-04']
TEST_YM  = ['2024-07', '2024-10']

train = feat[feat['ym'].isin(TRAIN_YM)]
test  = feat[feat['ym'].isin(TEST_YM)]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train['cascade']
X_test  = test[FEATURE_COLS].fillna(0)
y_test  = test['cascade']

print(f'Train : {len(X_train):,} rows | cascade rate: {y_train.mean():.3f}')
print(f'Test  : {len(X_test):,} rows  | cascade rate: {y_test.mean():.3f}')
print()
print('Class balance (train):')
print(y_train.value_counts(normalize=True).rename({0: 'no cascade', 1: 'cascade'}))

## 6. Train Models

In [ ]:
# Define three models
# class_weight='balanced' handles imbalanced classes automatically
# RF/GBM wrapped with CalibratedClassifierCV for well-calibrated probabilities (needed for agent)

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

rf_base = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    max_depth=8, random_state=42, n_jobs=-1
)
rf_pipe = CalibratedClassifierCV(rf_base, cv=3, method='isotonic')

gb_base = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.05,
    max_depth=4, subsample=0.8, random_state=42
)
gb_pipe = CalibratedClassifierCV(gb_base, cv=3, method='isotonic')

models = {
    'Logistic Regression': lr_pipe,
    'Random Forest'      : rf_pipe,
    'Gradient Boosting'  : gb_pipe,
}

results = {}
for name, model in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.4).astype(int)  # lower threshold: favour recall
    results[name] = {
        'model' : model,
        'y_prob': y_prob,
        'y_pred': y_pred,
        'AUC-ROC'  : roc_auc_score(y_test, y_prob),
        'AUC-PR'   : average_precision_score(y_test, y_prob),
        'F1'       : f1_score(y_test, y_pred),
        'Recall'   : recall_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
    }
    print(f'AUC={results[name]["AUC-ROC"]:.3f}  '
          f'Recall={results[name]["Recall"]:.3f}  '
          f'F1={results[name]["F1"]:.3f}')

In [ ]:
# ── Leakage / Overfit Validation ─────────────────────────────────────────────
# Two checks:
# 1. Train AUC vs Test AUC gap (overfit signal)
# 2. Ablation: remove granger_out_degree and see how much AUC drops

print('=== Check 1: Train vs Test AUC (overfit signal) ===')
for name, model in models.items():
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    test_auc  = results[name]['AUC-ROC']
    gap       = train_auc - test_auc
    flag      = '⚠ possible overfit' if gap > 0.02 else '✓ stable'
    print(f'  {name:25s} train={train_auc:.3f}  test={test_auc:.3f}  gap={gap:+.3f}  {flag}')

print()
print('=== Check 2: Ablation — remove granger_out_degree ===')
ABLATION_COLS = [c for c in FEATURE_COLS if c != 'granger_out_degree']

ablation_models = {
    'LR (no out_degree)' : Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'GBM (no out_degree)': CalibratedClassifierCV(
        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                    max_depth=4, subsample=0.8, random_state=42),
        cv=3, method='isotonic'
    ),
}

for name, model in ablation_models.items():
    model.fit(X_train[ABLATION_COLS], y_train)
    auc = roc_auc_score(y_test, model.predict_proba(X_test[ABLATION_COLS])[:, 1])
    f1  = f1_score(y_test, (model.predict_proba(X_test[ABLATION_COLS])[:, 1] >= 0.4).astype(int))
    print(f'  {name:30s} AUC={auc:.3f}  F1={f1:.3f}')

print()
print('Full model AUC (with out_degree):')
for name in ['Logistic Regression', 'Gradient Boosting']:
    print(f'  {name:30s} AUC={results[name]["AUC-ROC"]:.3f}')
print()
print('→ If ablation AUC drops significantly (>0.05), granger_out_degree was doing')
print('  most of the work (structural leakage).')
print('  If AUC stays high, dynamic features have genuine predictive power.')

## 7. Model Comparison

In [ ]:
# Metrics table
metrics_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ['model', 'y_prob', 'y_pred']}
    for name, res in results.items()
}).T.round(3)

print('=== Model Comparison ===')
print(metrics_df.to_string())
print()
best_name = metrics_df['AUC-ROC'].idxmax()
print(f'Best model (AUC-ROC): {best_name}')

In [ ]:
# ROC + PR curves
COLORS = {
    'Logistic Regression': '#4575b4',
    'Random Forest'      : '#d73027',
    'Gradient Boosting'  : '#1a9850',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC curve
ax = axes[0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, lw=2, color=COLORS[name],
             label=f'{name}  (AUC={res["AUC-ROC"]:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.4)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(fontsize=9)

# PR curve
ax = axes[1]
baseline = y_test.mean()
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res['y_prob'])
    ax.plot(rec, prec, lw=2, color=COLORS[name],
             label=f'{name}  (AUC-PR={res["AUC-PR"]:.3f})')
ax.axhline(baseline, color='gray', ls='--', lw=1,
             label=f'No-skill baseline ({baseline:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(fontsize=9)

plt.suptitle('Cascade Prediction — Model Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT/'web'/'assets'/'charts'/'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix for best model
best_res = results[best_name]
cm = confusion_matrix(y_test, best_res['y_pred'])

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['No Cascade', 'Cascade'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correct no-cascade): {tn:,}')
print(f'False Positives (false alarm)        : {fp:,}')
print(f'False Negatives (missed cascade)     : {fn:,}  ← want this low')
print(f'True Positives  (correct cascade)    : {tp:,}')

## 8. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression coefficients
lr_clf  = results['Logistic Regression']['model'].named_steps['clf']
lr_coef = pd.Series(lr_clf.coef_[0], index=FEATURE_COLS).sort_values()
colors  = ['#d73027' if v > 0 else '#4575b4' for v in lr_coef]
lr_coef.plot(kind='barh', ax=axes[0], color=colors, edgecolor='white')
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Logistic Regression — Coefficients\n(red = increases cascade risk)')
axes[0].set_xlabel('Coefficient')

# Random Forest: CalibratedClassifierCV stores fitted estimators in calibrated_classifiers_
# Average feature importances across all CV folds
rf_model = results['Random Forest']['model']
rf_importances = np.mean([
    cc.estimator.feature_importances_
    for cc in rf_model.calibrated_classifiers_
], axis=0)
rf_imp = pd.Series(rf_importances, index=FEATURE_COLS).sort_values(ascending=False)
rf_imp.plot(kind='bar', ax=axes[1], color='#1a9850', alpha=0.85, edgecolor='white')
axes[1].set_title('Random Forest — Feature Importance\n(avg across calibration folds)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Importance')

plt.suptitle('Feature Importance Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT/'web'/'assets'/'charts'/'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 3 features (LR magnitude):')
print(lr_coef.abs().sort_values(ascending=False).head(3))
print('\nTop 3 features (RF importance):')
print(rf_imp.head(3))

## 9. Save Best Model

In [ ]:
model_path = PROC_DIR / 'cascade_model.pkl'
payload = {
    'model'       : results[best_name]['model'],
    'model_name'  : best_name,
    'feature_cols': FEATURE_COLS,
    'metrics'     : {k: v for k, v in results[best_name].items()
                     if k not in ['model', 'y_prob', 'y_pred']},
    'threshold'   : 0.4,
}
with open(model_path, 'wb') as f:
    pickle.dump(payload, f)
print(f'Saved: {model_path}')
print(f'Model: {best_name}')
print(f'Metrics: { {k: round(v,3) for k,v in payload["metrics"].items()} }')

# ── Quick demo: Kenmore, 8am weekday, moderate delay ─────────────────────
# Kenmore betweenness=0.531, hist_bunching_rate=0.03, granger_out_degree=32
demo_cases = [
    {'label': 'Kenmore 8am weekday, moderate delay',
     'hour': 8,  'is_peak': 1, 'is_weekend': 0, 'delay_severity': 2,
     'betweenness': 0.531, 'hist_bunching_rate': 0.03,
     'granger_out_degree': 32, 'station_volume_norm': 0.8, 'n_trips': 6},
    {'label': 'Airport Blue 2pm weekend, mild delay',
     'hour': 14, 'is_peak': 0, 'is_weekend': 1, 'delay_severity': 1,
     'betweenness': 0.12, 'hist_bunching_rate': 0.01,
     'granger_out_degree': 5, 'station_volume_norm': 0.3, 'n_trips': 3},
]

print('\n=== Demo Predictions ===')
model = results[best_name]['model']
for case in demo_cases:
    row = pd.DataFrame([{
        'hour_sin'           : np.sin(2 * np.pi * case['hour'] / 24),
        'hour_cos'           : np.cos(2 * np.pi * case['hour'] / 24),
        'is_peak'            : case['is_peak'],
        'is_weekend'         : case['is_weekend'],
        'delay_severity'     : case['delay_severity'],
        'betweenness'        : case['betweenness'],
        'hist_bunching_rate' : case['hist_bunching_rate'],
        'granger_out_degree' : case['granger_out_degree'],
        'station_volume_norm': case['station_volume_norm'],
        'n_trips'            : case['n_trips'],
    }])
    prob = model.predict_proba(row)[0, 1]
    flag = '⚠ HIGH RISK' if prob >= 0.4 else '✓ low risk'
    print(f'  {case["label"]}')
    print(f'  → Cascade probability: {prob:.1%}  {flag}\n')